# Day 02 미니 프로젝트 · 방대한 문서로 실전 RAG 시스템 만들기 (경쟁형)
예시 몇 줄이 아니라 여러 문서로 된 지식베이스를 직접 청킹하고, 메타데이터로 필터링하고, 고급 검색과 지식그래프를 거쳐,
**LLM이 실제로 정답을 맞히는지**까지 채점합니다. 마지막에는 가장 좋은 조합을 찾아 최고 점수를 겨룹니다.

분량은 약 2시간 기준입니다.

### 진행
- Part 1 · 방대한 문서 → 청킹 + 메타데이터 → 벡터DB
- Part 2 · 메타데이터 필터링 검색
- Part 3 · 검색 기법 비교 · 리랭커는 왜 필요한가 · 검색 평가(Recall@k·MRR)
- Part 4 · GraphRAG — 지식그래프 작성·시각화·멀티홉
- Part 5 · 정답 품질 평가 — LLM 심판으로 정답률 채점
- Part 6 · 최적 조합 찾기 (대회) — 점수를 최대화하고 리더보드에 기록
- Part 7 · 최종 시스템 + 회고

### 대회 목표
`정답률`을 최대로 끌어올리는 검색·파라미터 조합을 찾는다. 가장 높은 점수가 우승.
---

## 0 · 설치

In [ ]:
!pip install -q sentence-transformers qdrant-client langchain-text-splitters openai rank_bm25 networkx matplotlib

## 1 · NVIDIA API 토큰 설정
생성과 채점(LLM 심판)에만 사용합니다. 검색·필터·평가 대부분은 로컬입니다.

In [ ]:
import os, getpass
from openai import OpenAI
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
_c = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
def ask(prompt, temperature=0.2, max_tokens=200):
    out = _c.chat.completions.create(model=LLM_MODEL, temperature=temperature, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]).choices[0].message.content
    return out.split("</think>")[-1].strip()
print("엔드포인트:", LLM_BASE_URL, "| 모델:", LLM_MODEL)

## Part 1 · 방대한 문서 확보
### 2 · 지식베이스 생성
정책 문서 몇 건과 템플릿으로 만든 여러 제품 매뉴얼을 합칩니다. 각 문서에 `doc_id·title·category·department·year` 메타데이터가 붙습니다.

In [ ]:
POLICY = [
 dict(title="연차 및 휴가 규정", category="인사", department="인사팀", year=2024, body=(
   "연차 휴가는 입사 1년 차부터 매년 15일이 부여되며 근속 연수에 따라 최대 25일까지 늘어난다. "
   "연차는 사용 3영업일 전까지 사내포털의 휴가 신청 메뉴에서 신청하고 팀장의 승인을 받아야 한다. "
   "반차와 반반차 단위 사용도 가능하며 동일한 절차를 따른다. 승인 없이 사용하면 결근으로 처리될 수 있다. "
   "승인된 휴가의 철회는 마이페이지의 휴가 내역에서 취소 요청한다.")),
 dict(title="비용 정산 절차", category="재무", department="재무팀", year=2024, body=(
   "비용 정산 신청은 지출일로부터 30일 이내에 완료해야 한다. 10만원 이상은 사전 승인이 필요하다. "
   "경비 청구는 사내포털의 정산 메뉴에서 접수하고 경영지원팀 검토 후 재무팀이 최종 승인한다. "
   "법인카드 사용 내역은 매월 5일까지 제출한다.")),
 dict(title="정보보안 정책", category="보안", department="보안팀", year=2025, body=(
   "외부 USB 등 이동식 저장장치의 사용은 원칙적으로 금지되며 파일 공유는 사내 승인된 드라이브를 통해서만 한다. "
   "외부 협력사와 자료 공유 시 보안팀의 사전 승인을 받아야 한다. 비밀번호는 설정 메뉴의 계정 보안에서 재설정하며 90일마다 변경한다.")),
 dict(title="재택근무 및 근태", category="인사", department="인사팀", year=2023, body=(
   "재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 사전 공유한다. 접속 시 VPN을 사용한다. "
   "코어타임은 오전 10시부터 오후 4시까지이며 이 시간에는 메신저 응답을 유지한다.")),
 dict(title="시스템 오류 코드 안내", category="IT", department="IT지원팀", year=2025, body=(
   "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다. ERR-5010은 네트워크 문제로 VPN을 확인한다. "
   "ERR-6003은 권한 부족으로 관리자에게 권한을 신청한다.")),
 dict(title="신입사원 온보딩", category="인사", department="인사팀", year=2024, body=(
   "입사 첫날 사내 계정과 장비를 지급하며 보안 서약서 작성이 필수다. 1주일 이내에 정보보안·성희롱 예방 교육을 이수한다. 수습 기간은 3개월이다.")),
]
suppliers = ["파워셀", "볼트원", "에너지코어"]
manuals = []
for n in range(100, 150):                         # 제품 50개 (서로 비슷해 검색이 헷갈리기 쉬움)
    pid = f"X-{n}"
    fw   = f"3.{n-100}"                            # 고유: 3.0 ~ 3.49
    volt = f"{3.0 + (n-100)*0.1:.1f}"              # 고유: 3.0 ~ 7.9
    mgr  = f"담당자{n-99}"                          # 고유: 담당자1 ~ 담당자50
    yr = 2022 + (n % 4)
    # 스펙을 한 덩어리(1청크)로 둡니다. 그래야 비슷한 제품끼리 헷갈려 정확 토큰 검색이 중요해집니다.
    manuals.append(dict(title=f"제품 {pid} 매뉴얼", category="제품", department="기술지원팀", year=yr, product_id=pid,
        body=(f"제품 {pid}의 펌웨어 최신 버전은 {fw}, 정격 전압은 {volt}V, "
              f"배터리 공급사는 {suppliers[n % 3]}, 기술 문의 담당자는 {mgr}이다.")))

documents = [dict(doc_id=f"D{i:03d}", **d) for i, d in enumerate(POLICY + manuals)]
print(f"문서 {len(documents)}건, 총 {sum(len(d['body']) for d in documents):,}자 | 카테고리:", sorted(set(d['category'] for d in documents)))

### 3 · 청킹 + 메타데이터 부착
각 문서 본문을 청킹하고, 모든 청크에 원본 메타데이터를 물려줍니다. 이 메타데이터가 필터의 기준입니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = []
for d in documents:
    for j, p in enumerate(splitter.split_text(d["body"])):
        meta = {k: d[k] for k in ("doc_id", "title", "category", "department", "year")}
        if "product_id" in d: meta["product_id"] = d["product_id"]
        chunks.append({"text": p, "chunk_idx": j, **meta})
print(f"총 청크 {len(chunks)}개 (문서 {len(documents)}건에서)")

### 4 · 임베딩 + 벡터DB (메타데이터를 payload로)

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
def embed(xs): return embedder.encode(xs, normalize_embeddings=True)
vecs = embed([c["text"] for c in chunks])
qc = QdrantClient(":memory:")
qc.recreate_collection("kb", vectors_config=VectorParams(size=vecs.shape[1], distance=Distance.COSINE))
qc.upsert("kb", points=[PointStruct(id=i, vector=vecs[i].tolist(), payload=chunks[i]) for i in range(len(chunks))])
print("저장 완료:", qc.count("kb").count, "청크")

## Part 2 · 메타데이터 필터링 검색
### 5 · 필터 없이 vs 필터로
메타데이터로 범위를 좁히면 원하는 문서만 검색됩니다.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range
def search(query, k=5, category=None, department=None, min_year=None):
    conds = []
    if category:   conds.append(FieldCondition(key="category",   match=MatchValue(value=category)))
    if department: conds.append(FieldCondition(key="department", match=MatchValue(value=department)))
    if min_year:   conds.append(FieldCondition(key="year",       range=Range(gte=min_year)))
    flt = Filter(must=conds) if conds else None
    pts = qc.query_points("kb", query=embed([query])[0].tolist(), limit=k, query_filter=flt).points
    return [(p.payload["category"], p.payload["year"], p.payload["text"][:32]) for p in pts]

print("[필터 없음] 펌웨어는 어떻게 업데이트하나요?")
for r in search("펌웨어는 어떻게 업데이트하나요?", 3): print("  ", r)
print("\n[category=제품 · 2024년 이후]")
for r in search("펌웨어는 어떻게 업데이트하나요?", 3, category="제품", min_year=2024): print("  ", r)

### 6 · [실습] 여러 필터 조합

In [ ]:
for label, kw in [("보안 카테고리만", dict(category="보안")),
                  ("IT지원팀만", dict(department="IT지원팀")),
                  ("2025년 이후만", dict(min_year=2025))]:
    print(f"[{label}] 인증 토큰이 만료되면?")
    for r in search("인증 토큰이 만료되면?", 2, **kw): print("  ", r)
    print()
# TODO: 내가 궁금한 조건을 조합해 검색해 보세요

## Part 3 · 검색 기법 비교
### 7 · BM25 · 하이브리드 · 리랭커

In [ ]:
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
def tok(t): return re.findall(r"[A-Za-z0-9\-]+|[가-힣]+", t)
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])
def vec_idx(query, n=10):  return [p.id for p in qc.query_points("kb", query=embed([query])[0].tolist(), limit=n).points]
def bm25_idx(query, n=10): s = bm25.get_scores(tok(query)); return sorted(range(len(s)), key=lambda i: -s[i])[:n]
def rrf(lists, k=60):
    sc = {}
    for ranks in lists:
        for r, i in enumerate(ranks): sc[i] = sc.get(i, 0) + 1/(k + r + 1)
    return sorted(sc, key=lambda i: -sc[i])
def hybrid_idx(query, n=10): return rrf([vec_idx(query, n), bm25_idx(query, n)])[:n]
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
def rerank_idx(query, n=3):
    cand = hybrid_idx(query, 12)
    sc = reranker.predict([(query, chunks[i]["text"]) for i in cand])
    return [cand[j] for j in sorted(range(len(cand)), key=lambda j: -sc[j])[:n]]
print("기법 준비: vec / bm25 / hybrid / rerank")

### 7.5 · 리랭커는 왜 필요한가 — 유사도만으로는 1등이 틀린다
아래 질문은 "어디에서 접수하는가"를 묻습니다. 임베딩 유사도는 단어가 더 겹치는 *기한* 문서를 1위로 올리지만(오답),
크로스인코더 리랭커는 (질문, 문서)를 함께 읽어 의도를 파악하고 *접수 장소* 문서를 1위로 올립니다.

In [ ]:
import numpy as np
q = "비용 정산은 어디에서 접수하나요?"
cands = [
    "비용 정산 신청은 지출일로부터 30일 이내에 완료해야 한다.",       # 단어는 더 겹치지만 '기한'
    "경비 청구는 사내포털의 정산 메뉴에서 접수하고 재무팀이 최종 승인한다.",  # 실제 '접수 장소'
    "재택근무는 주 2회까지 가능하다.",
]
sim = embed(cands) @ embed([q])[0]
rsc = reranker.predict([(q, c) for c in cands])
print("질문:", q)
print("\n[임베딩 유사도 순위]  ← 1위가 '기한' 문서(오답)")
for i in np.argsort(-sim): print(f"  {sim[i]:.3f}  {cands[i][:34]}")
print("\n[리랭커 순위]  ← 1위가 '접수 장소' 문서(정답)")
for i in np.argsort(-rsc): print(f"  {rsc[i]:6.2f}  {cands[i][:34]}")

### 8 · 검색 평가 — Recall@k · MRR
정답 키워드가 상위 청크에 들었는지로 검색 자체를 채점합니다.

In [ ]:
# 어려운 평가셋: 비슷한 제품 50개 중 정확히 그 제품을 찾아야 하고(정확 토큰),
# 단어만 겹치는 함정 문서에 흔들리지 않아야 정답이 나옵니다.
EVALSET = [
    ("X-137의 펌웨어 최신 버전은?",          "3.37",     "3.37"),
    ("X-142의 기술 문의 담당자는 누구인가요?", "담당자43", "담당자43"),
    ("X-119의 정격 전압은 몇 V인가요?",       "4.9",      "4.9V"),
    ("X-131의 펌웨어 버전은?",               "3.31",     "3.31"),
    ("인증 토큰이 만료되면 뜨는 코드는?",      "ERR-4021", "ERR-4021"),
    ("비용은 어디에서 접수하나요?",           "정산 메뉴", "사내포털의 정산 메뉴에서 접수한다"),
]
def eval_retrieval(fn, k=3):
    recall, mrr = 0, 0.0
    for q, kw, _ in EVALSET:
        idxs = fn(q, k)
        hit = [r for r, i in enumerate(idxs) if kw in chunks[i]["text"]]
        if hit: recall += 1; mrr += 1/(hit[0]+1)
    n = len(EVALSET); return round(recall/n, 3), round(mrr/n, 3)
print(f"{'기법':<12}{'Recall@3':>10}{'MRR':>8}")
print("-"*30)
for name, fn in [("벡터", vec_idx), ("BM25", bm25_idx), ("하이브리드", hybrid_idx), ("리랭커", rerank_idx)]:
    rec, mrr = eval_retrieval(fn); print(f"{name:<12}{rec:>10}{mrr:>8}")

## Part 4 · GraphRAG — 지식그래프 작성·시각화·멀티홉
벡터 검색은 답이 여러 문서에 흩어진 질문("X-200 제조사의 대표는?")에 약합니다.
관계를 트리플로 뽑아 그래프로 잇고, 경로를 따라 멀티홉으로 답합니다.
### 9 · 트리플 추출 → 그래프 작성

In [ ]:
import json, networkx as nx

graph_docs = [
    "제품 X-200은 아크메전자가 제조한다.",
    "아크메전자의 대표는 김철수다.",
    "X-200의 배터리는 파워셀이 공급한다.",
    "파워셀은 볼트원과 부품 공급 계약을 맺었다.",
    "김철수는 2024년 혁신경영 대상을 받았다.",
]
def extract_triples(docs):
    prompt = ('각 문장에서 (주체, 관계, 대상) 트리플을 뽑아 JSON 배열로만 출력해. 설명 금지.\n'
              '예: [["X-200","제조사","아크메전자"]]\n\n' + "\n".join(docs))
    raw = ask(prompt, temperature=0).strip().strip("`")
    if raw[:4].lower() == "json": raw = raw[4:].strip()
    return [tuple(t) for t in json.loads(raw)]

triples = extract_triples(graph_docs)
G = nx.DiGraph()
for s, r, o in triples: G.add_edge(s, o, rel=r)
print("트리플:", triples)
print("노드:", list(G.nodes))

### 10 · 그래프 시각화
작성한 지식그래프를 그림으로 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["AppleGothic", "Malgun Gothic", "NanumGothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(7, 5))
nx.draw(G, pos, with_labels=True, node_color="#DCE8FF", node_size=2600, font_size=11, edgecolors="#4A78D0", arrowsize=18)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "rel"), font_size=9)
plt.title("지식그래프"); plt.axis("off"); plt.show()

### 11 · 멀티홉 질의
질문 속 엔티티에서 경로를 따라가 답합니다. 한 문장에는 답이 없어도 경로로 연결됩니다.

In [ ]:
def neighbors(G, start, hops=2):
    seen, frontier, lines = {start}, {start}, []
    for _ in range(hops):
        nxt = set()
        for u in list(frontier):
            for _, v, d in G.out_edges(u, data=True): lines.append(f"{u} -[{d['rel']}]-> {v}"); nxt.add(v)
            for v, _, d in G.in_edges(u, data=True):  lines.append(f"{v} -[{d['rel']}]-> {u}"); nxt.add(v)
        frontier = nxt - seen; seen |= nxt
    return "\n".join(dict.fromkeys(lines))

def graph_answer(query):
    start = max((n for n in G.nodes if n in query), key=len, default=None)
    if not start: return "질문에서 그래프 엔티티를 찾지 못함"
    ctx = neighbors(G, start, 2)
    print(f"[{start}에서 2-hop 경로]\n{ctx}\n")
    return ask(f"[관계]\n{ctx}\n\n[질문] {query}\n관계만 근거로 짧게 답해줘.", temperature=0, max_tokens=80)

print("답:", graph_answer("X-200 제조사의 대표는 누구인가요?"))

## Part 5 · 정답 품질 평가 — LLM 심판
검색이 잘 됐는지(Recall/MRR)와, 최종 답이 실제로 맞았는지는 다른 문제입니다.
질문에 대해 RAG 파이프라인이 낸 답을, 기준 정답과 비교해 LLM이 채점합니다(맞으면 1, 아니면 0). 평균이 정답률입니다.
### 12 · 채점 함수

In [ ]:
SYSTEM = "아래 [자료]의 내용만 근거로 답하라. 없으면 '문서에 없음'이라고만 답하라."
def rag_answer(query, retriever, k=3):
    idxs = retriever(query, k)
    ctx = "\n".join(f"[{i+1}] {chunks[d]['text']}" for i, d in enumerate(idxs))
    return ask(f"{SYSTEM}\n[자료]\n{ctx}\n\n[질문] {query}", temperature=0.2, max_tokens=150)

def judge(question, reference, answer):
    p = (f"질문: {question}\n기준 정답: {reference}\nRAG 답변: {answer}\n"
         "RAG 답변이 기준 정답과 사실상 일치하면 1, 아니면 0. 숫자만 출력.")
    return "1" in ask(p, temperature=0, max_tokens=4)

def answer_accuracy(retriever, k=3, verbose=False):
    correct = 0
    for q, _, ref in EVALSET:
        a = rag_answer(q, retriever, k)
        ok = judge(q, ref, a); correct += ok
        if verbose: print(f"  [{'O' if ok else 'X'}] {q} -> {a[:40]}")
    return round(correct / len(EVALSET), 3)

print("하이브리드+리랭커 정답률:", answer_accuracy(rerank_idx, verbose=True))

## Part 6 · 최적 조합 찾기 (대회)
이제 목표는 하나입니다 — **정답률을 최대로**. 검색 기법과 파라미터(top_k 등)를 바꿔가며 가장 높은 점수의 조합을 찾습니다.
> 작은 데모 코퍼스에서는 여러 조합이 동점이 되기 쉽습니다. 평가셋을 늘리고 어려운 질문을 넣을수록 조합 간 차이가 커집니다.
### 13 · 조합별 정답률 비교

In [ ]:
configs = [
    ("벡터 · k=1",            lambda q, *_: vec_idx(q, 1)),
    ("벡터 · k=3",            lambda q, *_: vec_idx(q, 3)),
    ("하이브리드 · k=3",       lambda q, *_: hybrid_idx(q, 3)),
    ("하이브리드+리랭커 · k=1", lambda q, *_: rerank_idx(q, 1)),
    ("하이브리드+리랭커 · k=3", lambda q, *_: rerank_idx(q, 3)),
]
print(f"{'조합':<26}{'정답률':>8}")
print("-"*34)
board = []
for name, fn in configs:
    acc = answer_accuracy(fn)
    board.append((name, acc)); print(f"{name:<26}{acc:>8}")
best = max(board, key=lambda x: x[1])
if len({s for _, s in board}) == 1:
    print(f"\n모든 조합이 {best[1]}로 동점 — 코퍼스가 작고 질문이 쉬워서입니다.")
    print("평가셋을 늘리고 어려운 질문(정확 토큰·멀티홉·모호 질문)을 넣으면 조합 간 차이가 드러납니다.")
else:
    print(f"\n현재 최고: {best[0]} -> 정답률 {best[1]}")

### 14 · 리더보드 — 내 점수 기록
내가 찾은 최고의 조합과 점수를 리더보드에 올리세요. `chunk_size`, `top_k`, 필터, 기법을 바꿔 정답률을 더 끌어올려 보세요.
정답률이 가장 높은 사람이 우승입니다.

In [ ]:
LEADERBOARD = []
def submit(name, description, score):
    LEADERBOARD.append({"이름": name, "조합": description, "정답률": score})
    for row in sorted(LEADERBOARD, key=lambda r: -r["정답률"]):
        print(f"  {row['정답률']:>5}  {row['이름']:<10} {row['조합']}")

# 예시: 위에서 가장 좋았던 조합을 제출
submit("baseline", best[0], best[1])
# TODO: 내 이름으로 내 최고 조합을 제출하세요
# submit("홍길동", "하이브리드+리랭커, chunk_size=120, k=4", answer_accuracy(rerank_idx, k=4))

## Part 7 · 최종 시스템 + 회고
### 15 · 필터 + 최고 조합 + 생성
메타데이터 필터로 범위를 좁히고, 대회에서 이긴 조합으로 검색해 근거와 함께 답합니다.

In [ ]:
def filtered_idx(query, n=12, category=None, min_year=None):
    conds = []
    if category: conds.append(FieldCondition(key="category", match=MatchValue(value=category)))
    if min_year: conds.append(FieldCondition(key="year", range=Range(gte=min_year)))
    flt = Filter(must=conds) if conds else None
    return [p.id for p in qc.query_points("kb", query=embed([query])[0].tolist(), limit=n, query_filter=flt).points]

def final_rag(query, category=None, min_year=None, k=3):
    cand = filtered_idx(query, 12, category=category, min_year=min_year)
    if not cand: return "조건에 맞는 문서가 없습니다."
    sc = reranker.predict([(query, chunks[i]["text"]) for i in cand])
    best_idx = [cand[j] for j in sorted(range(len(cand)), key=lambda j: -sc[j])[:k]]
    ctx = "\n".join(f"[{i+1}] ({chunks[d]['title']}) {chunks[d]['text']}" for i, d in enumerate(best_idx))
    return ask(f"아래 [자료]만 근거로 답하고 없으면 '문서에 없음'. 답 끝에 근거 자료의 제목을 (출처: 제목)으로.\n[자료]\n{ctx}\n\n[질문] {query}",
               temperature=0.2, max_tokens=180)

print("Q: 연차는 며칠 전에 신청하나요?\nA:", final_rag("연차는 며칠 전에 신청하나요?"))
print("\nQ: X-105 펌웨어 어떻게 올려요? (category=제품)\nA:", final_rag("X-105 펌웨어 어떻게 올려요?", category="제품"))

### 16 · [내 문서 적용 + 회고]
- `POLICY`와 `manuals`를 내 실제 문서로 바꾸고(메타데이터 포함) 다시 실행하세요.
- 회고 3줄: ① 리랭커가 순위를 바로잡은 사례가 있었나 ② 어떤 조합이 정답률이 가장 높았나 ③ 어떤 질문에서 실패했나

## 산출물 체크리스트
- [ ] 여러 문서로 지식베이스를 만들고 메타데이터를 붙여 청킹했다
- [ ] 필터 있는/없는 검색 차이와, 리랭커가 순위를 바로잡는 사례를 확인했다
- [ ] 지식그래프를 작성·시각화하고 멀티홉으로 답했다
- [ ] LLM 심판으로 정답률을 채점하고, 조합을 바꿔 점수를 최대화했다
- [ ] 리더보드에 내 최고 점수를 기록했다